# Historical research notebook

Curated from the original collaborative PBL repository (source blob `a3a1665b3f1cc6fa37504fc2782c3558fcc30501`). Original cell order and research logic are retained; machine-specific PBL paths were replaced by paths relative to this notebook, and saved outputs/execution metadata were cleared for privacy and size. This is an archival notebook, not a verified end-to-end runnable pipeline. Original datasets, embeddings and clustering files are not included. Execute individual cells only after supplying the expected inputs and reviewing alternative branches. Some cells correspond to earlier experiments rather than the final configuration; see [data preparation notes](../../docs/data_preparation.md).


In [ ]:
import sys

sys.path.append('..')

In [ ]:
import pandas as pd
from src.preprocessing import *
from src.utils import *
from pymmseqs.commands import easy_cluster
from pathlib import Path

In [ ]:
df_raw = pd.read_csv('../data/raw/exp_enzymes.tsv', sep='\t')
df_one_ec_class = prepare_main_ec_classes(df_raw)
df_one_ec_class_nat = remove_unnatural_seqs(df_one_ec_class)
df_one_ec_class_nat_uniq = remove_duplicate_seqs(df_one_ec_class_nat)

In [ ]:
class_distribution(df_one_ec_class)

In [ ]:
min_seq_id = 0.3

df_dict = dict()
# First save to temporary fasta file

for i in range(1, 8):
    df_temp = df_one_ec_class_nat_uniq[df_one_ec_class_nat_uniq['EC main class'] == i]
    temp_fasta_path = Path(f'../temp/df_class{i}_temp.fasta')
    prefix = Path(f'../temp/df_class{i}')
    df_to_fasta(df_temp, temp_fasta_path)
    easy_cluster(fasta_files=temp_fasta_path, cluster_prefix=prefix, tmp_dir='../temp/', min_seq_id=min_seq_id)
    cmap_path = Path(f'{prefix}_cluster.tsv')
    df_dict[i] = cluster_df(df_temp, cmap_path)

df_clust_temp = pd.concat(df_dict.values(), ignore_index=True)
df_clust = remove_duplicate_seqs(df_clust_temp)

In [ ]:
df_to_tsv(df_clust, '../data/clustered/exp_enzymes_c30_exp.tsv')
df_to_fasta(df_clust, '../data/clustered/exp_enzymes_c30_exp.fasta')

In [ ]:
df_clust = pd.read_csv('../data/clustered/exp_enzymes_c30_exp.tsv', sep='\t')
min_seq_id = 0.3
usamp = ''

In [ ]:
df_final = preprocess_df(df_clust, flens=[2, 98], usamp=True, cap=2000, create_h5=True)